In [1]:
!pip install -q -U "huggingface_hub" "tokenizers" "datasets" "scikit-learn"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 84.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
transformers 5.0.0 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.23.1 which is incompatible.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.


In [2]:
import os
import re
import json
import random
import numpy as np
import pandas as pd

from kaggle_secrets import UserSecretsClient

from huggingface_hub import hf_hub_download

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

from sklearn.model_selection import train_test_split

from datasets import Dataset, DatasetDict

In [3]:
user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("HF-MTP-SLM")

print("Hugging Face token loaded successfully.")

Hugging Face token loaded successfully.


In [4]:
file_path = hf_hub_download(
    repo_id="L-NLProc/InLegalLlama-training-data",
    filename="For CPT/Train_data.zip",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Downloaded file:")
print(file_path)

For CPT/Train_data.zip: reconstructing file:   0%|          |  0.00B /  518MB            

For CPT/Train_data.zip: downloading bytes:           |  0.00B            

Downloaded file:
/root/.cache/huggingface/hub/datasets--L-NLProc--InLegalLlama-training-data/snapshots/91453101ebc33c4e49c9eb7376235aca1f18065b/For CPT/Train_data.zip


In [5]:
import zipfile

extract_dir = "/kaggle/working/legal_data"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(file_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("Dataset extracted to:")
print(extract_dir)

Dataset extracted to:
/kaggle/working/legal_data


In [6]:
for root, dirs, files in os.walk(extract_dir):
    for file in files:
        print(os.path.join(root, file))

/kaggle/working/legal_data/Train_data/train_for_llama2_cpt_SCI_38k_HC_100k_from_1800_2020.csv
/kaggle/working/legal_data/Train_data/.ipynb_checkpoints/Untitled-checkpoint.ipynb


In [7]:
DATA_DIR = "/kaggle/working/legal_data"

csv_path = None

for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.endswith(".csv"):
            csv_path = os.path.join(root, file)
            break

    if csv_path is not None:
        break

print("CSV:", csv_path)

CSV: /kaggle/working/legal_data/Train_data/train_for_llama2_cpt_SCI_38k_HC_100k_from_1800_2020.csv


In [8]:
df = pd.read_csv(csv_path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (138321, 1)
Columns: ['text']


In [9]:
print(df.info())

print("\nMissing values:")
print(df.isna().sum())

print("\nFirst document:")
print(df["text"].iloc[0][:1000])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138321 entries, 0 to 138320
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    138321 non-null  object
dtypes: object(1)
memory usage: 1.1+ MB
None

Missing values:
text    0
dtype: int64

First document:
A. Sharma, J.The petitioner has filed this writ application challenging the orders for the financial year passed by the Deputy Commissioner of Commercial Taxes, Tenughat Circle, Phusro, under the Bihar Finance Act, 1981 to as the Act) and the Central Sales Tax Act, 1956.Prayer for quashing the demand notices issued pursuant to the said orders, has also been made.It has also challenged the validity of Sub-section (3) of Section 45 of the Act whereby admission of appeal filed against an order has been barred, unless 20 per cent of the tax or the admitted tax whichever is greater is paid.The learned S.C.I., at the threshold, has raised preliminary objection about the of t

In [10]:
SEED = 61

CONTEXT_LENGTH = 2048

VOCAB_SIZE = 16000

MIN_FREQUENCY = 2

TRAIN_RATIO = 0.90

SPECIAL_TOKENS = [
    "<pad>",
    "<unk>",
    "<bos>",
    "<eos>",
    "<mask>"
]

In [11]:
def clean_text(text):
    
    if not isinstance(text, str):
        return ""
    
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)
    
    # Remove leading/trailing whitespace
    text = text.strip()
    
    return text

In [12]:
df["text"] = df["text"].map(clean_text)

In [13]:
before = len(df)

df = df[df["text"].str.len() > 0].copy()

df = df.reset_index(drop=True)

after = len(df)

print("Documents before:", before)
print("Documents after:", after)
print("Empty documents removed:", before - after)

Documents before: 138321
Documents after: 138321
Empty documents removed: 0


In [14]:
before = len(df)

df = df.drop_duplicates(
    subset=["text"]
).reset_index(drop=True)

after = len(df)

print("Documents before:", before)
print("Documents after:", after)
print("Duplicates removed:", before - after)

Documents before: 138321
Documents after: 133656
Duplicates removed: 4665


In [15]:
train_df, val_df = train_test_split(
    df,
    test_size=0.10,
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Training documents:", len(train_df))
print("Validation documents:", len(val_df))

Training documents: 120290
Validation documents: 13366


In [16]:
tokenizer = Tokenizer(
    BPE(unk_token="<unk>")
)

In [17]:
tokenizer.pre_tokenizer = ByteLevel()

In [18]:
trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=SPECIAL_TOKENS
)

In [19]:
def corpus_iterator():
    for text in train_df["text"]:
        yield text

In [20]:
tokenizer.train_from_iterator(
    corpus_iterator(),
    trainer=trainer
)

In [21]:
print(
    "Vocabulary size:",
    tokenizer.get_vocab_size()
)

Vocabulary size: 16000


In [22]:
sample_text = train_df["text"].iloc[0]

encoding = tokenizer.encode(sample_text)

print("Number of tokens:", len(encoding.ids))

print("\nFirst 50 tokens:")
print(encoding.tokens[:50])

print("\nFirst 50 token IDs:")
print(encoding.ids[:50])

Number of tokens: 10784

First 50 tokens:
['ĠCIVIL', 'ĠAPPELLATE', 'ĠCivil', 'ĠAppeals', 'ĠNos', '.', '196', 'Ġto', 'Ġ201', 'Ġof', 'Ġ1953', '.', 'Appeals', 'Ġfrom', 'Ġthe', 'Ġjudgment', 'Ġand', 'Ġof', 'Ġthe', 'ĠPunjab', 'ĠHigh', 'ĠCourt', 'Ġdated', 'Ġ30', ',', 'Ġ1949', ',', 'Ġin', 'ĠCivil', 'ĠRegular', 'ĠAppeals', 'ĠNos', '.', '15', '67', ',', 'Ġ15', '68', ',', 'Ġ15', '69', ',', 'Ġ15', '70', ',', 'Ġ15', '73', 'Ġan', 'Ġ15', '74']

First 50 token IDs:
[5022, 5554, 1102, 5889, 1398, 9, 2556, 100, 606, 83, 4379, 9, 5784, 241, 77, 499, 110, 83, 77, 1271, 376, 191, 365, 865, 7, 4138, 7, 99, 1102, 8142, 5889, 1398, 9, 1488, 4750, 7, 889, 4356, 7, 889, 5232, 7, 889, 4863, 7, 889, 4948, 160, 889, 4978]


In [23]:
legal_examples = [
    "Section 302 of the Indian Penal Code",
    "Article 226 of the Constitution of India",
    "Section 45(3) of the Bihar Finance Act, 1981",
    "AIR 1985 SC 330",
    "Cr.P.C.",
    "Supreme Court of India"
]

for text in legal_examples:
    
    encoding = tokenizer.encode(text)
    
    print("\nTEXT:")
    print(text)
    
    print("TOKENS:")
    print(encoding.tokens)


TEXT:
Section 302 of the Indian Penal Code
TOKENS:
['ĠSection', 'Ġ302', 'Ġof', 'Ġthe', 'ĠIndian', 'ĠPenal', 'ĠCode']

TEXT:
Article 226 of the Constitution of India
TOKENS:
['ĠArticle', 'Ġ226', 'Ġof', 'Ġthe', 'ĠConst', 'it', 'ution', 'Ġof', 'ĠIndia']

TEXT:
Section 45(3) of the Bihar Finance Act, 1981
TOKENS:
['ĠSection', 'Ġ45', '(', '3', ')', 'Ġof', 'Ġthe', 'ĠBihar', 'ĠFinance', 'ĠAct', ',', 'Ġ1981']

TEXT:
AIR 1985 SC 330
TOKENS:
['ĠAIR', 'Ġ1985', 'ĠSC', 'Ġ330']

TEXT:
Cr.P.C.
TOKENS:
['ĠCr', '.', 'P', '.', 'C', '.']

TEXT:
Supreme Court of India
TOKENS:
['ĠSupreme', 'ĠCourt', 'Ġof', 'ĠIndia']


In [24]:
TOKENIZER_DIR = "/kaggle/working/indian_legal_tokenizer"

os.makedirs(TOKENIZER_DIR, exist_ok=True)

tokenizer.save(
    os.path.join(
        TOKENIZER_DIR,
        "tokenizer.json"
    )
)

print("Tokenizer saved to:")
print(TOKENIZER_DIR)

Tokenizer saved to:
/kaggle/working/indian_legal_tokenizer


In [25]:
tokenizer_metadata = {
    "tokenizer_type": "BPE",
    "pre_tokenizer": "ByteLevel",
    "vocab_size": tokenizer.get_vocab_size(),
    "min_frequency": MIN_FREQUENCY,
    "special_tokens": SPECIAL_TOKENS,
    "training_documents": len(train_df),
    "context_length": CONTEXT_LENGTH,
    "seed": SEED
}

with open(
    os.path.join(
        TOKENIZER_DIR,
        "config.json"
    ),
    "w"
) as f:
    json.dump(
        tokenizer_metadata,
        f,
        indent=4
    )

print(json.dumps(tokenizer_metadata, indent=4))

{
    "tokenizer_type": "BPE",
    "pre_tokenizer": "ByteLevel",
    "vocab_size": 16000,
    "min_frequency": 2,
    "special_tokens": [
        "<pad>",
        "<unk>",
        "<bos>",
        "<eos>",
        "<mask>"
    ],
    "training_documents": 120290,
    "context_length": 2048,
    "seed": 61
}


In [26]:
PAD_ID = tokenizer.token_to_id("<pad>")
UNK_ID = tokenizer.token_to_id("<unk>")
BOS_ID = tokenizer.token_to_id("<bos>")
EOS_ID = tokenizer.token_to_id("<eos>")
MASK_ID = tokenizer.token_to_id("<mask>")

print("PAD:", PAD_ID)
print("UNK:", UNK_ID)
print("BOS:", BOS_ID)
print("EOS:", EOS_ID)
print("MASK:", MASK_ID)

PAD: 0
UNK: 1
BOS: 2
EOS: 3
MASK: 4


In [27]:
def tokenize_and_chunk(text):
    
    token_ids = tokenizer.encode(text).ids
    
    chunk_size = CONTEXT_LENGTH - 2
    
    chunks = []
    
    for i in range(
        0,
        len(token_ids),
        chunk_size
    ):
        
        chunk = token_ids[
            i:i + chunk_size
        ]
        
        # Ignore very small final fragments
        if len(chunk) < chunk_size // 2:
            continue
        
        chunk = (
            [BOS_ID]
            + chunk
            + [EOS_ID]
        )
        
        chunks.append(chunk)
    
    return chunks

In [28]:
test_text = train_df["text"].iloc[0]

test_chunks = tokenize_and_chunk(test_text)

print("Original token count:")
print(len(tokenizer.encode(test_text).ids))

print("\nNumber of chunks:")
print(len(test_chunks))

for i, chunk in enumerate(test_chunks):
    print(
        f"Chunk {i + 1}: {len(chunk)} tokens"
    )

Original token count:
10784

Number of chunks:
5
Chunk 1: 2048 tokens
Chunk 2: 2048 tokens
Chunk 3: 2048 tokens
Chunk 4: 2048 tokens
Chunk 5: 2048 tokens


In [29]:
decoded_text = tokenizer.decode(
    test_chunks[0]
)

print(decoded_text[:2000])

ĠCIVIL ĠAPPELLATE ĠCivil ĠAppeals ĠNos . 196 Ġto Ġ201 Ġof Ġ1953 . Appeals Ġfrom Ġthe Ġjudgment Ġand Ġof Ġthe ĠPunjab ĠHigh ĠCourt Ġdated Ġ30 , Ġ1949 , Ġin ĠCivil ĠRegular ĠAppeals ĠNos . 15 67 , Ġ15 68 , Ġ15 69 , Ġ15 70 , Ġ15 73 Ġan Ġ15 74 Ġof Ġ1942 Ġarising Ġout Ġof Ġthe Ġdated ĠJuly Ġ31 , Ġ1942 , Ġof Ġthe ĠCourt Ġof Ġthe ĠDistrict ĠJudge , ĠHoshiarpur Ġin ĠAppeals ĠNos . 104 / 35 Ġof Ġ1941 - Ġ42 , 101 / 32 Ġof Ġ1941 , Ġ103 / 34 Ġ- of Ġ1941 / 42 ) Ġ15 / 73 Ġof Ġ1941 , Ġ102 / 33 Ġof Ġ1941 / 42 Ġand Ġ120 Ġof Ġ1941 Ġarising Ġout Ġof Ġthe Ġdated ĠJuly Ġ24 , Ġ1941 , Ġof Ġthe ĠCourt Ġof ĠSubordinate ĠJudge , Ġ4 th ĠClass , ĠK ang ra Ġin ĠSu its ĠNos . 54 4 , Ġ5 48 , Ġ5 45 , Ġ5 47 , Ġ5 46 Ġand Ġ5 49 Ġof Ġ1940 . B ang ĠBe har ilal Ġand ĠK . ĠR . ĠChaudh ury , Ġfor Ġthe Ġappellant . G an pat ĠRai , Ġfor Ġthe Ġrespondent . M . ĠSik ri , Ġfor ĠPunjab , ĠJ indra ĠLal Ġand ĠR . ĠDhe bar , Ġfor Ġthe Ġ( State Ġof ĠPunjab ). 1956 . O ct ober Ġ23 . The ĠJudgment Ġof Ġthe ĠCourt Ġwas Ġby ĠK . ĠD AS ĠJ 

In [30]:
train_chunks = []

for i, text in enumerate(train_df["text"]):
    
    chunks = tokenize_and_chunk(text)
    
    train_chunks.extend(chunks)
    
    if (i + 1) % 5000 == 0:
        print(
            f"Processed {i + 1:,} / "
            f"{len(train_df):,} documents | "
            f"Chunks: {len(train_chunks):,}"
        )

Processed 5,000 / 120,290 documents | Chunks: 6,327
Processed 10,000 / 120,290 documents | Chunks: 12,665
Processed 15,000 / 120,290 documents | Chunks: 19,112
Processed 20,000 / 120,290 documents | Chunks: 25,576
Processed 25,000 / 120,290 documents | Chunks: 32,000
Processed 30,000 / 120,290 documents | Chunks: 38,457
Processed 35,000 / 120,290 documents | Chunks: 45,012
Processed 40,000 / 120,290 documents | Chunks: 51,547
Processed 45,000 / 120,290 documents | Chunks: 57,987
Processed 50,000 / 120,290 documents | Chunks: 64,410
Processed 55,000 / 120,290 documents | Chunks: 70,727
Processed 60,000 / 120,290 documents | Chunks: 77,321
Processed 65,000 / 120,290 documents | Chunks: 83,924
Processed 70,000 / 120,290 documents | Chunks: 90,408
Processed 75,000 / 120,290 documents | Chunks: 96,992
Processed 80,000 / 120,290 documents | Chunks: 103,250
Processed 85,000 / 120,290 documents | Chunks: 109,805
Processed 90,000 / 120,290 documents | Chunks: 116,161
Processed 95,000 / 120,290 

In [31]:
val_chunks = []

for i, text in enumerate(val_df["text"]):
    
    chunks = tokenize_and_chunk(text)
    
    val_chunks.extend(chunks)
    
    if (i + 1) % 2000 == 0:
        print(
            f"Processed {i + 1:,} / "
            f"{len(val_df):,} validation documents | "
            f"Chunks: {len(val_chunks):,}"
        )

Processed 2,000 / 13,366 validation documents | Chunks: 2,550
Processed 4,000 / 13,366 validation documents | Chunks: 5,175
Processed 6,000 / 13,366 validation documents | Chunks: 7,793
Processed 8,000 / 13,366 validation documents | Chunks: 10,394
Processed 10,000 / 13,366 validation documents | Chunks: 12,970
Processed 12,000 / 13,366 validation documents | Chunks: 15,701


In [32]:
train_dataset = Dataset.from_dict({
    "input_ids": train_chunks
})

val_dataset = Dataset.from_dict({
    "input_ids": val_chunks
})

In [33]:
print(train_dataset)
print(val_dataset)

Dataset({
    features: ['input_ids'],
    num_rows: 155060
})
Dataset({
    features: ['input_ids'],
    num_rows: 17474
})


In [34]:
processed_dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})

print(processed_dataset)

DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 155060
    })
    validation: Dataset({
        features: ['input_ids'],
        num_rows: 17474
    })
})


In [35]:
sample_lengths = [
    len(x)
    for x in train_dataset["input_ids"][:1000]
]

print("Minimum:", min(sample_lengths))
print("Maximum:", max(sample_lengths))
print("Mean:", np.mean(sample_lengths))

Minimum: 1029
Maximum: 2048
Mean: 1863.881


In [36]:
example = train_dataset[0]["input_ids"]

print("Number of tokens:", len(example))

print("\nFirst 20 IDs:")
print(example[:20])

print("\nLast 20 IDs:")
print(example[-20:])

Number of tokens: 2048

First 20 IDs:
[2, 5022, 5554, 1102, 5889, 1398, 9, 2556, 100, 606, 83, 4379, 9, 5784, 241, 77, 499, 110, 83, 77]

Last 20 IDs:
[424, 1002, 100, 197, 11662, 264, 7, 110, 14212, 3259, 1228, 2112, 253, 568, 100, 77, 14212, 3259, 1228, 3]


In [37]:
print(
    tokenizer.decode(example)
)

ĠCIVIL ĠAPPELLATE ĠCivil ĠAppeals ĠNos . 196 Ġto Ġ201 Ġof Ġ1953 . Appeals Ġfrom Ġthe Ġjudgment Ġand Ġof Ġthe ĠPunjab ĠHigh ĠCourt Ġdated Ġ30 , Ġ1949 , Ġin ĠCivil ĠRegular ĠAppeals ĠNos . 15 67 , Ġ15 68 , Ġ15 69 , Ġ15 70 , Ġ15 73 Ġan Ġ15 74 Ġof Ġ1942 Ġarising Ġout Ġof Ġthe Ġdated ĠJuly Ġ31 , Ġ1942 , Ġof Ġthe ĠCourt Ġof Ġthe ĠDistrict ĠJudge , ĠHoshiarpur Ġin ĠAppeals ĠNos . 104 / 35 Ġof Ġ1941 - Ġ42 , 101 / 32 Ġof Ġ1941 , Ġ103 / 34 Ġ- of Ġ1941 / 42 ) Ġ15 / 73 Ġof Ġ1941 , Ġ102 / 33 Ġof Ġ1941 / 42 Ġand Ġ120 Ġof Ġ1941 Ġarising Ġout Ġof Ġthe Ġdated ĠJuly Ġ24 , Ġ1941 , Ġof Ġthe ĠCourt Ġof ĠSubordinate ĠJudge , Ġ4 th ĠClass , ĠK ang ra Ġin ĠSu its ĠNos . 54 4 , Ġ5 48 , Ġ5 45 , Ġ5 47 , Ġ5 46 Ġand Ġ5 49 Ġof Ġ1940 . B ang ĠBe har ilal Ġand ĠK . ĠR . ĠChaudh ury , Ġfor Ġthe Ġappellant . G an pat ĠRai , Ġfor Ġthe Ġrespondent . M . ĠSik ri , Ġfor ĠPunjab , ĠJ indra ĠLal Ġand ĠR . ĠDhe bar , Ġfor Ġthe Ġ( State Ġof ĠPunjab ). 1956 . O ct ober Ġ23 . The ĠJudgment Ġof Ġthe ĠCourt Ġwas Ġby ĠK . ĠD AS ĠJ 

In [38]:
PROCESSED_DIR = "/kaggle/working/indian_legal_2048"

processed_dataset.save_to_disk(
    PROCESSED_DIR
)

print("Processed dataset saved to:")
print(PROCESSED_DIR)

Saving the dataset (0/3 shards):   0%|          | 0/155060 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/17474 [00:00<?, ? examples/s]

Processed dataset saved to:
/kaggle/working/indian_legal_2048


In [39]:
metadata = {
    "source": "L-NLProc/InLegalLlama-training-data",
    "source_file": "For CPT/Train_data.zip",
    "num_documents": len(df),
    "train_documents": len(train_df),
    "validation_documents": len(val_df),
    "train_sequences": len(train_dataset),
    "validation_sequences": len(val_dataset),
    "vocab_size": tokenizer.get_vocab_size(),
    "context_length": CONTEXT_LENGTH,
    "tokenizer": "BPE",
    "pre_tokenizer": "ByteLevel",
    "seed": SEED
}

with open(
    os.path.join(
        PROCESSED_DIR,
        "metadata.json"
    ),
    "w"
) as f:
    json.dump(
        metadata,
        f,
        indent=4
    )

print(json.dumps(metadata, indent=4))

{
    "source": "L-NLProc/InLegalLlama-training-data",
    "source_file": "For CPT/Train_data.zip",
    "num_documents": 133656,
    "train_documents": 120290,
    "validation_documents": 13366,
    "train_sequences": 155060,
    "validation_sequences": 17474,
    "vocab_size": 16000,
    "context_length": 2048,
    "tokenizer": "BPE",
    "pre_tokenizer": "ByteLevel",
    "seed": 61
}


In [40]:
print("=" * 60)
print("INDIAN LEGAL SLM — PREPROCESSING COMPLETE")
print("=" * 60)

print(f"Documents:          {len(df):,}")
print(f"Training documents: {len(train_df):,}")
print(f"Validation docs:    {len(val_df):,}")
print(f"Vocabulary size:    {tokenizer.get_vocab_size():,}")
print(f"Context length:     {CONTEXT_LENGTH}")
print(f"Train sequences:    {len(train_dataset):,}")
print(f"Val sequences:      {len(val_dataset):,}")
print(f"Tokenizer:          BPE")
print(f"Saved to:           {PROCESSED_DIR}")

INDIAN LEGAL SLM — PREPROCESSING COMPLETE
Documents:          133,656
Training documents: 120,290
Validation docs:    13,366
Vocabulary size:    16,000
Context length:     2048
Train sequences:    155,060
Val sequences:      17,474
Tokenizer:          BPE
Saved to:           /kaggle/working/indian_legal_2048
